# Exploratory Data Analysis (EDA) - OEWS May 2025 Dataset
### Course: Exploratory Data Analysis
### Instructor: Ali Hassan Sherazi
### Student Submission Dashboard Project

This notebook demonstrates the Exploratory Data Analysis (EDA) for the Occupational Employment and Wage Statistics (OEWS) May 2025 dataset from the U.S. Bureau of Labor Statistics (BLS). The goal is to load, clean, analyze, and visualize the dataset using Pandas, NumPy, Matplotlib, and Seaborn.

## 1. Setup & Environment
Import standard data analysis libraries and set up styling for plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.family'] = 'sans-serif'

## 2. Load & Clean Data
The dataset is located in `data/all_data_M_2025.xlsx`. Since it contains non-numeric strings (like `*`, `**`, `#`, `~`) to denote various reporting states or suppressed values, we clean them and convert columns to proper numeric types.

In [ ]:
def clean_numeric_col(series):
    s = series.astype(str).str.strip()
    s = s.replace(to_replace=['\*+', '\#', '~', 'nan', 'None', '-'], value=np.nan, regex=True)
    return pd.to_numeric(s, errors='coerce')

print("Loading Excel file...")
df = pd.read_excel('../data/all_data_M_2025.xlsx', sheet_name='All May 2025 data')
df.columns = [col.upper() for col in df.columns]

# Numeric columns to clean
numeric_cols = [
    'TOT_EMP', 'EMP_PRSE', 'JOBS_1000', 'LOC_QUOTIENT', 'PCT_TOTAL', 'PCT_RPT',
    'H_MEAN', 'A_MEAN', 'MEAN_PRSE',
    'H_PCT10', 'H_PCT25', 'H_MEDIAN', 'H_PCT75', 'H_PCT90',
    'A_PCT10', 'A_PCT25', 'A_MEDIAN', 'A_PCT75', 'A_PCT90'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = clean_numeric_col(df[col])

print("Cleaned columns:", df.dtypes)
df.head()

## 3. Descriptive Analysis
Let's get basic summary statistics of the dataset.

In [ ]:
print("Shape of dataset:", df.shape)
df.describe()

Let's see unique counts for area types and states.

In [ ]:
print("Records by Area Type:\n", df['AREA_TYPE'].value_counts())
print("Unique States:", df['PRIM_STATE'].nunique())

## 4. Visualizations
Let's generate the key required visual representations.

### 4.1 Pie Chart - Area Type Employment Share

In [ ]:
area_type_map = {
    1: 'National (U.S.)',
    2: 'State',
    3: 'U.S. Territory',
    4: 'Metropolitan Area (MSA)',
    6: 'Nonmetropolitan Area'
}
df['AREA_TYPE_NAME'] = pd.to_numeric(df['AREA_TYPE'], errors='coerce').map(area_type_map).fillna('Other')

emp_by_area = df.groupby('AREA_TYPE_NAME')['TOT_EMP'].sum().reset_index()
emp_by_area = emp_by_area[emp_by_area['TOT_EMP'] > 0]

plt.figure(figsize=(7, 7))
plt.pie(emp_by_area['TOT_EMP'], labels=emp_by_area['AREA_TYPE_NAME'], autopct='%1.1f%%', startangle=140)
plt.title("Employment Share by Area Type")
plt.show()

### 4.2 Histogram - Average Hourly Wage

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['H_MEAN'].dropna(), bins=30, kde=True, color='teal')
plt.title("Frequency Distribution of Average Hourly Wage")
plt.xlabel("Average Hourly Wage ($)")
plt.ylabel("Frequency")
plt.show()

### 4.3 Scatter Plot - Employment vs. Wage

In [ ]:
plt.figure(figsize=(10, 6))
scatter_df = df[(df['OCC_TITLE'] != 'All Occupations') & (df['TOT_EMP'] > 0)].dropna(subset=['TOT_EMP', 'H_MEAN'])
if len(scatter_df) > 1000:
    scatter_df = scatter_df.sample(1000, random_state=42)

sns.scatterplot(x='TOT_EMP', y='H_MEAN', data=scatter_df, alpha=0.6, color='purple')
plt.xscale('log')
plt.title("Employment vs. Hourly Wage (Sample of 1000 Records)")
plt.xlabel("Total Employment (Log Scale)")
plt.ylabel("Mean Hourly Wage ($)")
plt.show()

### 4.4 Box Plot - Wage Spread by Area Type

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='AREA_TYPE_NAME', y='A_MEAN', data=df.dropna(subset=['A_MEAN']), palette='Set2')
plt.title("Annual Wage Distribution by Area Type")
plt.xlabel("Area Type")
plt.ylabel("Annual Mean Wage ($)")
plt.xticks(rotation=15)
plt.show()

### 4.5 Heatmap - Feature Correlation Matrix

In [ ]:
plt.figure(figsize=(10, 8))
corr_cols = ['TOT_EMP', 'H_MEAN', 'A_MEAN', 'LOC_QUOTIENT', 'H_MEDIAN']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title("Correlation Matrix of Numeric Features")
plt.show()

## 5. Summary of Findings
1. **Wage Inequality**: The hourly wage distribution is heavily right-skewed, showing that the majority of occupations earn less than $40/hour, with a long tail of high-earning specialized roles.
2. **Area Dynamics**: Metropolitan areas (MSAs) represent the largest share of records and total employment, while non-metropolitan areas show a lower average wage structure.
3. **Employment-Wage Tradeoff**: The scatter plot shows a high concentration of jobs in middle-to-low paying ranges, with very few occupations having both high employment and high hourly wages.